Note: Code template was originally taken from https://www.kaggle.com/code/utm529fg/scraping-restaurant-reviews-from-tabelog, with changes made to account for tabelog website changes since 2023 and to ensure closed restaurants were not included

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import polars as pl
from tqdm.notebook import tqdm

In [5]:
def get_pref_url_list(url):
    ret = []
    r = requests.get(url + 'rstLst/')
    soup = BeautifulSoup(r.content, 'html.parser')
    soup_url_list = soup.find_all('a', class_='list-balloon__recommend-target')
    url_dict = {}
    for soup_url in soup_url_list:
        url = soup_url.get('href')
        if not 'rstLst' in url:
            area = soup_url.contents[0]
            area = area.replace(' ','').replace('\n','')
            ret.append([area,url])
    return ret

def get_muni_url_list(url):
    ret = []
    r = requests.get(url + 'rstLst/')
    soup = BeautifulSoup(r.content, 'html.parser')
    soup_url_list = soup.find_all('a', class_='c-link-arrow')
    for soup_url in soup_url_list:
        url = soup_url.get('href')
        if not 'span' in soup_url and url[-7:]=='rstLst/':
            area = soup_url.span.string
            area = area.replace(' ','').replace('\n','')
            ret.append([area,url])
    return ret

def get_muni2_url_list(url):
    ret = []
    r = requests.get(url)
    soup = BeautifulSoup(r.content, 'html.parser')
    soup_url_list = soup.find_all('a', class_='c-link-arrow')
    for soup_url in soup_url_list:
        url = soup_url.get('href')
        if url[-7:]=='rstLst/':
            area = soup_url.span.string
            area = area.replace(' ','').replace('\n','')
            ret.append([area,url])
    return ret

In [9]:
def get_review(url, area_list):
    col = ['store_id','name','nearest_station','nearest_distance','genre',
           'rating_val','review_cnt','save_cnt',
           'budget_dinner','budget_lunch','holiday','address',
           'prefecture','municipalities_1','municipalities_2','municipalities_3']
    df = pl.DataFrame([])

    for page in range(1, 61):
        r = requests.get(url + str(page))
        soup = BeautifulSoup(r.content, 'html.parser')

        soup_num = soup.find_all('span', class_='c-page-count__num')

        # If pagination widget is missing (blocked/empty/different layout), stop gracefully
        if len(soup_num) < 2:
            # Optional quick debug:
            # print("No page-count nums:", r.status_code, r.url, soup.title.get_text(strip=True) if soup.title else None)
            break

        start = int(soup_num[0].get_text())
        end = int(soup_num[1].get_text())

        if start == 0 and end == 0:
            break

        num = end - start + 1

        # Your "Over 1,200" check, but only if soup_num[2] exists
        if end == 1200 and len(soup_num) > 2:
            try:
                if int(soup_num[2].get_text()) > 1200:
                    print('Over 1,200')
            except ValueError:
                pass

        soup_list = soup.find_all('div', class_='list-rst__wrap js-open-new-window')
        if not soup_list:
            break

        # Don’t index past what’s actually on the page
        num = min(num, len(soup_list))

        for i in range(num):
            soup_tmp = soup_list[i].find('a', class_='list-rst__rst-name-target cpy-rst-name')
            if soup_tmp is None:
                continue

            store_id = soup_tmp.get('href').split('/')[6]
            name = soup_tmp.get_text().strip()

            soup_tmp = soup_list[i].find('div', class_='list-rst__area-genre cpy-area-genre')
            soup_tmp = soup_tmp.get_text() if soup_tmp else ""
            parts = soup_tmp.split(' / ')
            nearest = parts[0] if len(parts) > 0 else "-"
            genre = parts[1].replace('\n','') if len(parts) > 1 else "-"

            if ' ' in nearest and 'm' in nearest:
                distance = nearest.split(' ')[2].replace('m','')
                nearest = nearest.split(' ')[1]
            else:
                distance = 'NaN'
                nearest = '-'
            distance = float(distance) if distance != 'NaN' else float("nan")

            rating_val = soup_list[i].find('span', class_='c-rating__val c-rating__val--strong list-rst__rating-val')
            rating_val = rating_val.get_text().strip() if rating_val else 'NaN'
            rating_val = 'NaN' if rating_val == '-' else rating_val
            rating_val = float(rating_val) if rating_val != 'NaN' else float("nan")

            review_cnt = soup_list[i].find('em', class_='list-rst__rvw-count-num cpy-review-count')
            review_cnt = review_cnt.get_text().strip() if review_cnt else 'NaN'
            review_cnt = 'NaN' if review_cnt == '-' else review_cnt
            review_cnt = float(review_cnt) if review_cnt != 'NaN' else float("nan")

            save_cnt = soup_list[i].find('span', class_='list-rst__save-count-num u-text-num')
            save_cnt = save_cnt.get_text().strip() if save_cnt else 'NaN'
            save_cnt = 'NaN' if save_cnt == '-' else save_cnt
            save_cnt = float(save_cnt) if save_cnt != 'NaN' else float("nan")

            # Budget: sometimes missing or only one value
            soup_tmp = soup_list[i].find_all('span', class_='c-rating-v3__val')
            budget_dinner = soup_tmp[0].get_text().strip() if len(soup_tmp) > 0 else '-'
            budget_lunch  = soup_tmp[1].get_text().strip() if len(soup_tmp) > 1 else '-'

            holiday = soup_list[i].find('span', class_='list-rst__holiday-text')
            holiday = holiday.get_text().strip() if holiday else '-'

            addr_tag = soup_list[i].find('p', class_='list-rst__address cpy-address')
            address = addr_tag.get_text().strip() if addr_tag else '-'

            df_tmp = pl.DataFrame([[store_id,name,nearest,distance,genre,
                                    rating_val,review_cnt,save_cnt,
                                    budget_dinner,budget_lunch,holiday,address] + area_list],
                                  schema=col)
            df = pl.concat([df, df_tmp])

    return df

<h3>
    If you specify a location outside of Tokyo, the process may not be completed within 12 hours.<br>
    Please divide the process or focus on the necessary areas.
</h3>

In [20]:
url_top = 'https://tabelog.com/'
target_area = '東京'

# scraping 47 prefectures url list
# ex https://tabelog.com/tokyo
pref_url_list = get_pref_url_list(url_top)
area = ['']*4
df = pl.DataFrame([])

for pref_url in pref_url_list:
    area[0] = pref_url[0]
    
    # filter Tokyo
    if area[0]==target_area:
        
        # scraping the 1st level municipalities
        # ex https://tabelog.com/tokyo/C13101/rstLst/
        muni_url_list = get_muni_url_list(pref_url[1])
        for muni_url in muni_url_list:
            area[1] = muni_url[0]
            muni2_url_list = get_muni2_url_list(muni_url[1])
            
            # scraping the 2nd level municipalities
            # ex https://tabelog.com/tokyo/C13101/C36030/rstLst/
            for muni2_url in muni2_url_list:
                area[2] = muni2_url[0]
                # There is no 3rd level municipalities
                if muni2_url[1].count('/')==7:
                    area[3] = '-'
                    print(*area,muni2_url[1])
                    df = pl.concat([df,get_review(muni2_url[1],area)])
                # scraping the 3rd level municipalities
                else:
                    muni3_url_list = get_muni2_url_list(muni2_url[1])
                    for muni3_url in muni3_url_list:
                        area[3] = muni3_url[0]
                        print(*area,muni3_url[1])
                        df = pl.concat([df,get_review(muni3_url[1],area)])

東京 千代田区 飯田橋 - https://tabelog.com/tokyo/C13101/C36030/rstLst/


C:\Users\mckha\AppData\Local\Temp\ipykernel_13556\2416439999.py:91: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_tmp = pl.DataFrame([[store_id,name,nearest,distance,genre,


東京 千代田区 一番町 - https://tabelog.com/tokyo/C13101/C36031/rstLst/
東京 千代田区 岩本町 - https://tabelog.com/tokyo/C13101/C36032/rstLst/
東京 千代田区 内神田 - https://tabelog.com/tokyo/C13101/C36033/rstLst/
東京 千代田区 内幸町 - https://tabelog.com/tokyo/C13101/C36034/rstLst/
東京 千代田区 大手町 - https://tabelog.com/tokyo/C13101/C36035/rstLst/
東京 千代田区 鍛冶町 - https://tabelog.com/tokyo/C13101/C36037/rstLst/
東京 千代田区 霞が関 - https://tabelog.com/tokyo/C13101/C36038/rstLst/
東京 千代田区 神田相生町 - https://tabelog.com/tokyo/C13101/C36040/rstLst/
東京 千代田区 神田淡路町 - https://tabelog.com/tokyo/C13101/C36041/rstLst/
東京 千代田区 神田和泉町 - https://tabelog.com/tokyo/C13101/C36042/rstLst/
東京 千代田区 神田岩本町 - https://tabelog.com/tokyo/C13101/C36043/rstLst/
東京 千代田区 神田小川町 - https://tabelog.com/tokyo/C13101/C36044/rstLst/
東京 千代田区 神田鍛冶町 - https://tabelog.com/tokyo/C13101/C36045/rstLst/
東京 千代田区 神田北乗物町 - https://tabelog.com/tokyo/C13101/C36046/rstLst/
東京 千代田区 神田紺屋町 - https://tabelog.com/tokyo/C13101/C36047/rstLst/
東京 千代田区 神田佐久間河岸 - https://tabelog.com/tokyo/C13101/C3

In [ ]:
# url_top = 'https://tabelog.com/'
# target_area = '東京'

# # scraping 47 prefectures url list
# # ex https://tabelog.com/tokyo
# pref_url_list = get_pref_url_list(url_top)
# area = ['']*4
# df = pl.DataFrame([])

# for pref_url in pref_url_list:
#     area[0] = pref_url[0]
    
#     # filter Tokyo
#     if area[0]==target_area:
        
#         # scraping the 1st level municipalities
#         # ex https://tabelog.com/tokyo/C13101/rstLst/
#         muni_url_list = get_muni_url_list(pref_url[1])
#         muni_url = muni_url_list[1]
        
#         area[1] = muni_url[0]
#         muni2_url_list = get_muni2_url_list(muni_url[1])
        
#         # scraping the 2nd level municipalities
#         # ex https://tabelog.com/tokyo/C13101/C36030/rstLst/
#         for muni2_url in muni2_url_list:
#             area[2] = muni2_url[0]
#             # There is no 3rd level municipalities
#             if muni2_url[1].count('/')==7:
#                 area[3] = '-'
#                 print(*area,muni2_url[1])
#                 df = pl.concat([df,get_review(muni2_url[1],area)])
#             # scraping the 3rd level municipalities
#             else:
#                 muni3_url_list = get_muni2_url_list(muni2_url[1])
#                 for muni3_url in muni3_url_list:
#                     area[3] = muni3_url[0]
#                     print(*area,muni3_url[1])
#                     df = pl.concat([df,get_review(muni3_url[1],area)])

東京 中央区 明石町 - https://tabelog.com/tokyo/C13102/C36100/rstLst/


C:\Users\mckha\AppData\Local\Temp\ipykernel_13556\2416439999.py:91: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_tmp = pl.DataFrame([[store_id,name,nearest,distance,genre,


東京 中央区 入船 - https://tabelog.com/tokyo/C13102/C36101/rstLst/
東京 中央区 勝どき - https://tabelog.com/tokyo/C13102/C36102/rstLst/
東京 中央区 京橋 - https://tabelog.com/tokyo/C13102/C36103/rstLst/
東京 中央区 銀座 - https://tabelog.com/tokyo/C13102/C36104/rstLst/
Over 1,200
東京 中央区 新川 - https://tabelog.com/tokyo/C13102/C36105/rstLst/
東京 中央区 新富 - https://tabelog.com/tokyo/C13102/C36106/rstLst/
東京 中央区 月島 - https://tabelog.com/tokyo/C13102/C36107/rstLst/
東京 中央区 築地 - https://tabelog.com/tokyo/C13102/C36108/rstLst/
東京 中央区 佃 - https://tabelog.com/tokyo/C13102/C36109/rstLst/
東京 中央区 豊海町 - https://tabelog.com/tokyo/C13102/C36110/rstLst/
東京 中央区 日本橋 - https://tabelog.com/tokyo/C13102/C36111/rstLst/
東京 中央区 日本橋大伝馬町 - https://tabelog.com/tokyo/C13102/C36112/rstLst/
東京 中央区 日本橋蛎殻町 - https://tabelog.com/tokyo/C13102/C36113/rstLst/
東京 中央区 日本橋兜町 - https://tabelog.com/tokyo/C13102/C36114/rstLst/
東京 中央区 日本橋茅場町 - https://tabelog.com/tokyo/C13102/C36115/rstLst/
東京 中央区 日本橋小網町 - https://tabelog.com/tokyo/C13102/C36116/rstLst/
東京 中央区 

In [21]:
df

store_id,name,nearest_station,nearest_distance,genre,rating_val,review_cnt,save_cnt,budget_dinner,budget_lunch,holiday,address,prefecture,municipalities_1,municipalities_2,municipalities_3
str,str,str,f64,str,f64,f64,f64,str,str,str,str,str,str,str,str
"""13202412""","""飯田橋 喝采""","""飯田橋駅""",189.0,"""日本料理、居酒屋、創作料理 """,3.46,269.0,25540.0,"""￥8,000～￥9,999""","""-""","""-""","""東京都千代田区飯田橋3-11-20 SPビル 1F""","""東京""","""千代田区""","""飯田橋""","""-"""
"""13306936""","""鳥酒処""","""飯田橋駅""",447.0,"""居酒屋、焼き鳥、もつ鍋 """,3.12,7.0,516.0,"""￥4,000～￥4,999""","""～￥999""","""-""","""東京都千代田区飯田橋1-7-8 梶山ビル 1F""","""東京""","""千代田区""","""飯田橋""","""-"""
"""13011164""","""平川""","""飯田橋駅""",380.0,"""日本料理、海鮮、すき焼き """,3.41,104.0,4735.0,"""￥8,000～￥9,999""","""￥5,000～￥5,999""","""-""","""東京都千代田区飯田橋3-10-8 ホテルメトロポリタンエドモ…","""東京""","""千代田区""","""飯田橋""","""-"""
"""13178488""","""貸切 ダイニングバー Rupo""","""飯田橋駅""",277.0,"""ダイニングバー、イタリアン """,3.31,72.0,3373.0,"""￥4,000～￥4,999""","""～￥999""","""-""","""東京都千代田区飯田橋3-3-11 ばんらいビル １Ｆ""","""東京""","""千代田区""","""飯田橋""","""-"""
"""13114542""","""和牛焼肉食べ放題 肉屋の台所 飯田橋店""","""飯田橋駅""",62.0,"""焼肉、ビュッフェ、居酒屋 """,3.15,224.0,14004.0,"""￥4,000～￥4,999""","""￥1,000～￥1,999""","""-""","""東京都千代田区飯田橋4-9-8 飯田橋プラザ8F""","""東京""","""千代田区""","""飯田橋""","""-"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""13272155""","""アイランドリゾート母島 ナンプー""","""-""",NaN,"""旅館・民宿 """,NaN,NaN,11.0,"""-""","""-""","""-""","""東京都小笠原村母島字元地""","""東京""","""小笠原村""","""母島""","""-"""
"""13131218""","""クラフト イン ラメーフ""","""-""",NaN,"""旅館・民宿 """,3.13,9.0,60.0,"""￥4,000～￥4,999""","""-""","""-""","""東京都小笠原村母島字静沢""","""東京""","""小笠原村""","""母島""","""-"""
"""13272183""","""ドルフィン 別館""","""-""",NaN,"""その他 """,NaN,NaN,7.0,"""-""","""-""","""-""","""東京都小笠原村母島字元地""","""東京""","""小笠原村""","""母島""","""-"""


In [22]:
df.write_csv('Tokyo_Restaurant_Reviews_Tabelog.csv')

In [24]:
len(df) 

131082